In [ ]:
import numpy as np
import scipy.io.wavfile as wav
from scipy.signal import stft

#Backend_DFT 

def read_audio(file_path):
    """
    Membaca file WAV dan mengembalikan data sinyal serta frekuensi sampling.
    """
    sample_rate, data = wav.read(file_path)
    
    # Jika stereo, ambil satu channel saja
    if len(data.shape) > 1:
        data = data[:, 0]
    
    # Normalisasi amplitudo ke rentang [-1, 1]
    data = data / np.max(np.abs(data))
    return data, sample_rate


def compute_dft(signal):
    """"
    Menghitung DFT (Discrete Fourier Transform) menggunakan rumus dasar.
    Rumus DFT:
        X[k] = Σ (n=0 → N-1) x[n] * e^(-j * 2π * k * n / N)
    """
    N = len(signal)
    n = np.arange(N)
    k = n.reshape((N, 1))
    # Matriks eksponensial kompleks
    M = np.exp(-2j * np.pi * k * n / N)
    X = np.dot(M, signal)
    return X


def compute_fft(signal):
    """
    Menghitung FFT (Fast Fourier Transform) versi cepat dari DFT.
    """
    return np.fft.fft(signal)


def compute_spectrogram(signal, sample_rate, window_size=1024, overlap=512):
    """
    Menghitung spektrogram menggunakan STFT (Short-Time Fourier Transform).
    Rumus STFT:
        X(m, ω) = Σ x[n] * w[n - mR] * e^(-j * ω * n)
    dimana:
        w[n] = window function
        R = hop size (perpindahan antar jendela)
    """
    f, t, Zxx = stft(signal, fs=sample_rate, nperseg=window_size, noverlap=overlap)
    magnitude = np.abs(Zxx)
    return f, t, magnitude


def analyze_audio(file_path): #digunakan untuk menggabungkan dan membaca semua fungsi diatas    
    
    # 1. Baca file
    signal, sr = read_audio(file_path)

    # 2. Hitung DFT dan FFT
    dft_result = compute_dft(signal[:2048])  # ambil sebagian sinyal biar cepat
    fft_result = compute_fft(signal)

    # 3. Hitung spektrogram
    f, t, spec = compute_spectrogram(signal, sr)

    # 4. Kembalikan semua hasil
    return {
        "signal": signal,
        "sample_rate": sr,
        "DFT": dft_result,
        "FFT": fft_result,
        "frequencies": f,
        "times": t,
        "spectrogram": spec
    }
